# Amazon Nova Act SDK를 활용한 Browser tool Live View

## 개요

이 튜토리얼에서는 Nova Act SDK로 Amazon Bedrock AgentCore Browser tool과 상호 작용하고 브라우저 화면을 실시간으로 확인하는 방법을 알아봅니다.


### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| Agent 유형          | 단일                                                                             |
| Agentic Framework   | Nova Act                                                                         |
| LLM 모델            | Amazon Nova Act model                                                            |
| 튜토리얼 구성 요소  | NovaAct를 사용해 Browser tool과 실시간으로 상호 작용                             |
| 튜토리얼 분야       | 여러 분야                                                                        |
| 예제 난이도         | 쉬움                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK, Nova Act                                     |

### 튜토리얼 아키텍처

이 튜토리얼에서는 Nova Act와 Browser tool을 함께 사용하면서 브라우저 화면을 실시간으로 확인하는 방법을 설명합니다.  

예제에서는 Nova Act Agent에 자연어 지시를 보내 Bedrock AgentCore Browser에서 작업을 수행하고, 그 과정을 실시간으로 확인합니다.

<div style="text-align:left">
    <img src="./images/browser-tool.png" width="50%"/>
</div>

### 튜토리얼 주요 기능

* Browser tool을 사용하고 화면을 실시간으로 확인
* Nova Act와 Browser tool을 함께 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.10+
* AWS 자격 증명. IAM 역할/사용자에 다음 권한이 있어야 합니다. https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* Nova Act SDK 및 API 키

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Live View로 NovaAct와 Bedrock AgentCore Browser tool 함께 사용하기

여기서는 helper function을 사용해 Amazon DCV SDK를 통해 Bedrock AgentCore Browser tool에 연결합니다.




In [ ]:
%%writefile live_view_with_nova_act.py
from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct
from rich.console import Console
from rich.panel import Panel
import sys
import json
import argparse
sys.path.append("../interactive_tools")
from browser_viewer import BrowserViewerServer

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("using region", region)

def live_view_with_nova_act(prompt, starting_page, nova_act_key, region="us-west-2"):
    """화면 크기를 설정해 브라우저 라이브 뷰어를 실행합니다."""
    console.print(
        Panel(
            "[bold cyan]Browser Live Viewer[/bold cyan]\n\n"
            "This demonstrates:\n"
            "• Live browser viewing with DCV\n"
            "• Configurable display sizes (not limited to 900×800)\n"
            "• Proper display layout callbacks\n\n"
            "[yellow]Note: Requires Amazon DCV SDK files[/yellow]",
            title="Browser Live Viewer",
            border_style="blue",
        )
    )

    try:
        # 1단계: 브라우저 세션 생성
        with browser_session(region) as client:
            ws_url, headers = client.generate_ws_headers()

            # 2단계: Viewer server 시작
            console.print("\n[cyan]Step 3: Starting viewer server...[/cyan]")
            viewer = BrowserViewerServer(client, port=8000)
            viewer_url = viewer.start(open_browser=True)

            # 3단계: 기능 표시
            console.print("\n[bold green]Viewer Features:[/bold green]")
            console.print(
                "• Default display: 1600×900 (configured via displayLayout callback)"
            )
            console.print("• Size options: 720p, 900p, 1080p, 1440p")
            console.print("• Real-time display updates")
            console.print("• Take/Release control functionality")

            console.print("\n[yellow]Press Ctrl+C to stop[/yellow]")

            # 4단계: Nova Act를 사용해 브라우저와 상호 작용
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=starting_page,
            ) as nova_act:
                result = nova_act.act(prompt)
                console.print(f"\n[bold green]Nova Act Result:[/bold green] {result}")
        
    except Exception as e:
        console.print(f"\n[red]Error: {e}[/red]")
        import traceback
        traceback.print_exc()
    finally:
        console.print("\n\n[yellow]Shutting down...[/yellow]")
        if "client" in locals():
            client.stop()
            console.print("✅ Browser session terminated")
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Browser Search instruction")
    parser.add_argument("--starting-page", required=True, help="Starting URL")
    parser.add_argument("--nova-act-key", required=True, help="Nova Act API key")
    parser.add_argument("--region", default="us-west-2", help="AWS region")
    args = parser.parse_args()

    result = live_view_with_nova_act(
        args.prompt, args.starting_page, args.nova_act_key, args.region
    )

    with open('result.txt', 'w') as f:
        f.write(str(result))

    console.print(f"\n[bold green]Nova Act Result:[/bold green] {result}")

#### 스크립트 실행하기
스크립트를 실행하기 전에 아래에 Nova Act API 키를 입력합니다.

In [ ]:
NOVA_ACT_KEY = ""  ### 여기에 Nova Act API 키를 입력하세요

In [ ]:
!python live_view_with_nova_act.py --prompt "Search for macbooks and extract the details of the first one" --starting-page "https://www.amazon.com/" --nova-act-key {NOVA_ACT_KEY}

### 내부에서는 어떤 일이 일어났을까요? 
* Browser client를 인스턴스화하고 세션을 시작했습니다.
* 그런 다음 `BrowserViewerServer`를 사용해 브라우저 세션에 연결하고 로컬에서 세션 화면을 확인했습니다.
* Nova Act Agent를 생성하고 브라우저 세션 정보를 전달했습니다.
* 이어서 Nova Act Agent에 자연어 지시를 보내고 수행되는 액션을 실시간으로 확인했습니다.

## 브라우저에서 CAPTCHA 처리하기
다음으로 브라우저에서 CAPTCHA를 처리하는 방법을 살펴보겠습니다. CAPTCHA는 웹 사이트와 상호 작용하는 주체가 bot이 아니라 사람인지 확인하기 위한 것입니다. 따라서 Agent가 CAPTCHA를 해결하도록 하지 않고, 사용자가 직접 제어권을 가져와 CAPTCHA를 해결한 후 Agent가 계속 작업하도록 합니다.

새 스크립트를 생성해 보겠습니다. Nova Act를 사용하면 페이지에 CAPTCHA가 있는지 확인할 수 있습니다. 이 기능을 활용해 Nova Act가 계속 작업하기 전에 CAPTCHA를 처리합니다.

#### 스크립트가 실행되면 브라우저의 로컬 화면이 표시됩니다. 실행 중 CAPTCHA가 나타나면 직접 해결하세요. 스크립트는 CAPTCHA가 해결될 때까지 기다립니다.

##### 참고: CAPTCHA가 나타나지 않고 스크립트가 정상적으로 완료되면 CAPTCHA가 나타날 때까지 스크립트를 다시 실행해 보세요.

In [ ]:
%%writefile captcha_with_nova_act.py
from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct, BOOL_SCHEMA, ActAgentError
from rich.console import Console
from rich.panel import Panel
import sys
import json
import time
import argparse
sys.path.append("../interactive_tools")
from browser_viewer import BrowserViewerServer


console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("using region", region)

def contains_human_validation_error(err):
    """
    오류 또는 오류의 message 속성이 HumanValidationError를 나타내는지 재귀적으로 확인합니다.
    """
    if err is None:
        return False

    # 문자열을 직접 확인
    if isinstance(err, str) and "HumanValidationError" in err:
        return True

    # err의 'message' 속성이 문자열 또는 다른 오류이면 재귀적으로 확인
    if hasattr(err, "message"):
        return contains_human_validation_error(err.message)

    # err의 문자열 표현에 오류 텍스트가 포함되어 있는지 확인
    if "HumanValidationError" in str(err):
        return True

    return False

def live_view_with_nova_act(steps, starting_page, nova_act_key, region="us-west-2"):
    """화면 크기를 설정해 브라우저 라이브 뷰어를 실행합니다."""
    console.print(
        Panel(
            "[bold cyan]Browser Live Viewer[/bold cyan]\n\n"
            "This demonstrates:\n"
            "• Live browser viewing with DCV\n"
            "• Configurable display sizes (not limited to 900×800)\n"
            "• Proper display layout callbacks\n\n"
            "[yellow]Note: Requires Amazon DCV SDK files[/yellow]",
            title="Browser Live Viewer",
            border_style="blue",
        )
    )
    result = None

    try:
        # 1단계: 브라우저 세션 생성
        with browser_session(region) as client:
            ws_url, headers = client.generate_ws_headers()

            # 2단계: Viewer server 시작
            console.print("\n[cyan]Step 3: Starting viewer server...[/cyan]")
            viewer = BrowserViewerServer(client, port=8000)
            viewer_url = viewer.start(open_browser=True)

            # 3단계: 기능 표시
            console.print("\n[bold green]Viewer Features:[/bold green]")
            console.print(
                "• Default display: 1600×900 (configured via displayLayout callback)"
            )
            console.print("• Size options: 720p, 900p, 1080p, 1440p")
            console.print("• Real-time display updates")
            console.print("• Take/Release control functionality")

            console.print("\n[yellow]Press Ctrl+C to stop[/yellow]")

            # 4단계: Nova Act를 사용해 브라우저와 상호 작용
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=starting_page,
            ) as nova_act:
                
                for step_index, step in enumerate(steps):
                    max_retries = 3
                    retry_count = 0
                    
                    while retry_count < max_retries:
                        try:
                            print(f"Executing step {step_index + 1}/{len(steps)}: {step}")
                            result = nova_act.act(step)
                            console.print(f"\n[bold green]Step {step_index + 1} Result:[/bold green] {result}")
                            break  # 성공하면 다음 단계로 이동
                            
                        except ActAgentError as err:
                            # 오류 메시지 또는 구조에서 사람의 검증이 필요한지 확인
                            if contains_human_validation_error(err):
                                print("CAPTCHA detected! Please solve it in the browser.")
                                captcha_wait_attempts = 0
                                max_captcha_wait_attempts = 8
                                
                                while captcha_wait_attempts < max_captcha_wait_attempts:
                                    try:
                                        time.sleep(10)  # 사용자가 CAPTCHA를 해결할 시간 제공
                                        captcha_result = nova_act.act(
                                            "Is there a captcha on the screen?", schema=BOOL_SCHEMA
                                        )
                                        
                                        if captcha_result.matches_schema and not captcha_result.parsed_response:
                                            print("Captcha solved, continuing with current step...")
                                            # 현재 단계를 불이익 없이 재시도하도록 retry_count를 증가시키지 않음
                                            break
                                        else:
                                            print(f"Captcha still present. Waiting... (Attempt {captcha_wait_attempts + 1}/{max_captcha_wait_attempts})")
                                            captcha_wait_attempts += 1
                                            
                                    except Exception as captcha_check_err:
                                        print(f"Error checking captcha status: {str(captcha_check_err)}")
                                        captcha_wait_attempts += 1
                                        time.sleep(5)
                                
                                if captcha_wait_attempts >= max_captcha_wait_attempts:
                                    print("Maximum captcha wait attempts reached. Trying to continue anyway.")
                                    retry_count += 1
                                
                            else:
                                print(f"Non-captcha error occurred: {str(err)}")
                                retry_count += 1
                                time.sleep(5)
                                
                        except Exception as general_err:
                            print(f"Unexpected error on step {step_index + 1}: {str(general_err)}")
                            retry_count += 1
                            time.sleep(5)
                            
                    if retry_count >= max_retries:
                        console.print(f"\n[bold red]Failed to complete step {step_index + 1} after {max_retries} attempts.[/bold red]")
                        if step_index < len(steps) - 1:
                            console.print("[yellow]Attempting to continue with next step...[/yellow]")
                
                # 최종 요약
                console.print("\n[bold blue]Task Execution Complete[/bold blue]")
        
    except Exception as e:
        console.print(f"\n[red]Error: {e}[/red]")
        import traceback
        traceback.print_exc()
    finally:
        console.print("\n\n[yellow]Shutting down...[/yellow]")
        if "client" in locals():
            client.stop()
            console.print("✅ Browser session terminated")
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--steps", required=False, help="JSON array or comma-separated list of steps to execute", 
                        default='["Search for AI news and press enter. If there is alredy AI news typed in search bar, then do not do anything", "Get the first AI news result, open the page and extract the title. Instead, if you see an AI summary, extract the first paragrpah of the summary and return"]')
    parser.add_argument("--starting-page", required=True, help="Starting URL")
    parser.add_argument("--nova-act-key", required=True, help="Nova Act API key")
    parser.add_argument("--region", default="us-west-2", help="AWS region")
    args = parser.parse_args()

    # steps 파싱: JSON 배열 또는 쉼표로 구분된 값 허용
    try:
        # 먼저 JSON으로 파싱 시도
        steps = json.loads(args.steps)
    except json.JSONDecodeError:
        # 유효한 JSON이 아니면 쉼표로 구분된 문자열로 처리
        steps = [step.strip() for step in args.steps.split(',')]

    # steps가 list인지 확인
    if not isinstance(steps, list):
        steps = [steps]

    result = live_view_with_nova_act(
        steps, args.starting_page, args.nova_act_key, args.region
    )

In [ ]:
!python captcha_with_nova_act.py  --starting-page "https://www.google.com/" --nova-act-key {NOVA_ACT_KEY}

# 축하합니다!